In [ ]:
!uv pip install numpy matplotlib torch transformers scikit-learn

In [ ]:
import sys
sys.path.insert(0, "/Users/cretuluca/uni/ro-doc-classification")

from src.utils.config import load_config, get_device
from src.data.dataset import load_data
from src.training.full import load_model, train
from src.eval.metrics import count_parameters, evaluate, measure_latency, save_results

config = load_config()
device = get_device()

In [ ]:
splits, tokenizer = load_data(config)

In [ ]:
model = load_model(config)
trainable_params, total_params = count_parameters(model)

In [ ]:
trainer = train(model, splits["train"]["dataset"], splits["validation"]["dataset"], config)

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_loss = [e["loss"] for e in log if "loss" in e and "eval_loss" not in e]
val_loss = [e["eval_loss"] for e in log if "eval_loss" in e]
val_acc = [e["eval_accuracy"] for e in log if "eval_accuracy" in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_loss, label="Train")
ax1.plot(val_loss, label="Validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()

ax2.plot(val_acc, label="Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")
ax2.legend()

plt.tight_layout()
plt.savefig("outputs/full/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
_, _, report_dict, cm = evaluate(trainer.model, splits["test"]["dataset"], device, config["training"]["full"]["batch_size"])

In [ ]:
max_len = config["model"]["max_length"]
sample_text = splits["test"]["samples"][0]
latency_ms = measure_latency(trainer.model, tokenizer, sample_text, device, max_len)

In [ ]:
save_results("outputs/full", config, trainer, report_dict, cm, latency_ms, trainable_params, total_params)